In [0]:
%sql
-- Cria o schema (dataset)
CREATE SCHEMA IF NOT EXISTS workspace.nyc_taxi

In [0]:
%sql
-- Cria os volumes de armazenamento dos dados brutos
CREATE VOLUME IF NOT EXISTS workspace.nyc_taxi.landing_zone;

In [0]:
# Os arquivos foram inseridos na landing zone manualmente
# Verifica os arquivos na landing zone
display(dbutils.fs.ls("/Volumes/workspace/nyc_taxi/landing_zone/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/nyc_taxi/landing_zone/yellow_tripdata_2023-01.parquet,yellow_tripdata_2023-01.parquet,47673370,1778385211000
dbfs:/Volumes/workspace/nyc_taxi/landing_zone/yellow_tripdata_2023-02.parquet,yellow_tripdata_2023-02.parquet,47748012,1778385211000
dbfs:/Volumes/workspace/nyc_taxi/landing_zone/yellow_tripdata_2023-03.parquet,yellow_tripdata_2023-03.parquet,56127762,1778385213000
dbfs:/Volumes/workspace/nyc_taxi/landing_zone/yellow_tripdata_2023-04.parquet,yellow_tripdata_2023-04.parquet,54222699,1778385213000
dbfs:/Volumes/workspace/nyc_taxi/landing_zone/yellow_tripdata_2023-05.parquet,yellow_tripdata_2023-05.parquet,58654627,1778385214000


In [0]:
# Define os diretórios da landing zone e da consumption zone
LANDING_PATH = "/Volumes/workspace/nyc_taxi/landing_zone/"


In [0]:
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql import functions as f

# Lê os arquivos parquet da landing zone e define o schema explicitamente das colunas necessárias para que não haja inconsistências nos tipos dos campos

# Lê cada arquivo separadamente
dfs = []
for month in ["01", "02", "03", "04", "05"]:
    path = f"{LANDING_PATH}yellow_tripdata_2023-{month}.parquet"
    df = spark.read.parquet(path)

    # Renomeia Airport_fee para airport_fee se necessário
    if "Airport_fee" in df.columns:
        df = df.withColumnRenamed("Airport_fee", "airport_fee")

    # Seleciona e casteia as colunas obrigatórias com os tipos corretos
    df = df.select(
        f.col("VendorID").cast("long"),
        f.col("passenger_count").cast("integer"),
        f.col("total_amount").cast("double"),
        f.col("tpep_pickup_datetime").cast("timestamp"),
        f.col("tpep_dropoff_datetime").cast("timestamp"),
    )
    dfs.append(df)

# Une todos os meses em um único DataFrame
df_final = reduce(DataFrame.union, dfs)

print(f"Total de linhas: {df_final.count():,}")
df_final.printSchema()

Total de linhas: 16,186,386
root
 |-- VendorID: long (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)



In [0]:
# Salva os arquivos como Delta Table no catálogo
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nyc_taxi.yellow_trips")

In [0]:
%sql
-- Consultando a tabela para validacão
SELECT * FROM workspace.nyc_taxi.yellow_trips LIMIT 10

VendorID,passenger_count,total_amount,tpep_pickup_datetime,tpep_dropoff_datetime
2,1,14.3,2023-01-01T00:32:10.000Z,2023-01-01T00:40:36.000Z
2,1,16.9,2023-01-01T00:55:08.000Z,2023-01-01T01:01:27.000Z
2,1,34.9,2023-01-01T00:25:04.000Z,2023-01-01T00:37:49.000Z
1,0,20.85,2023-01-01T00:03:48.000Z,2023-01-01T00:13:25.000Z
2,1,19.68,2023-01-01T00:10:29.000Z,2023-01-01T00:21:19.000Z
2,1,27.8,2023-01-01T00:50:34.000Z,2023-01-01T01:02:52.000Z
2,1,20.52,2023-01-01T00:09:22.000Z,2023-01-01T00:19:49.000Z
2,1,64.44,2023-01-01T00:27:12.000Z,2023-01-01T00:49:56.000Z
2,1,28.38,2023-01-01T00:21:44.000Z,2023-01-01T00:36:40.000Z
2,1,19.9,2023-01-01T00:39:42.000Z,2023-01-01T00:50:36.000Z


In [0]:
%sql
-- Adiciona descrição da tabela
ALTER TABLE workspace.nyc_taxi.yellow_trips 
SET TBLPROPERTIES (
    'comment' = 'Tabela com dados de corridas de táxi em Nova York para o ano de 2023.'
);

-- Adiciona descrição das colunas
ALTER TABLE workspace.nyc_taxi.yellow_trips
ALTER COLUMN VendorID
COMMENT 'Código do fornecedor (1: Creative Mobile Technologies, 2: Curb Mobility, 6: Myle Technologies, 7: Helix).';

ALTER TABLE workspace.nyc_taxi.yellow_trips
ALTER COLUMN passenger_count
COMMENT 'Número de passageiros no veículo.';

ALTER TABLE workspace.nyc_taxi.yellow_trips
ALTER COLUMN total_amount
COMMENT 'Valor total cobrado do passageiro.';

ALTER TABLE workspace.nyc_taxi.yellow_trips
ALTER COLUMN tpep_pickup_datetime
COMMENT 'Data e hora em que o taxímetro foi acionado.';

ALTER TABLE workspace.nyc_taxi.yellow_trips
ALTER COLUMN tpep_dropoff_datetime
COMMENT 'Data e hora em que o taxímetro foi desligado.';